# B01 · Supervised reference: ViT-B/16 fine-tuned on building crops (development events)

How far are the frozen probes from a model **trained for the task**? A ViT-B/16 (ImageNet-21k + 1k) is
fine-tuned end to end on 32 m crops centred on each building (112 px), one model per held-out development event,
trained on every other development and secondary event. Settings are fixed in
`src/damagesar_bench/baselines/finetune.py` (2 epochs, AdamW 2e-5, batch 64, class-balanced loss, seed 0) and
never tuned on a held-out event. **Exploratory reference, not a contribution.**

Compared with the p03 frozen probes (DINOv3 and ImageNet ViT-B, outline pooling) on the same events, with the
same event-level statistics.

* **Before running:** Runtime → Change runtime type → **T4 GPU**. Secrets: `GH_TOKEN` (the model is public).
  Needs p01 (building table) and p03 (its run folder).
* **Time:** about 30–45 minutes (crops ~5 min, 5 folds × 2 epochs); about 2 compute units.
* **Outputs:** `fold_scores.parquet`, `event_metrics.csv`, `comparisons.csv` in this run's folder.

In [ ]:
NOTEBOOK_NAME = "b01_supervised_baseline"

# Drive layout (FYP Google account). Keep in sync with src/damagesar_bench/drive.py
import os, datetime, re
from google.colab import drive
drive.mount("/content/drive")
PROJECT = "/content/drive/MyDrive/DamageSAR-Bench"
FOLDERS = ["00_admin", "01_data/raw/bright", "01_data/raw/instance_labels", "01_data/external",
           "01_data/derived", "02_models/checkpoints", "02_models/features", "03_notebooks/pipeline",
           "03_notebooks/checks", "03_notebooks/scratch", "04_outputs/runs", "04_outputs/reports"]
for f in FOLDERS:
    os.makedirs(f"{PROJECT}/{f}", exist_ok=True)
def run_dir(name):
    slug = re.sub(r"[^a-z0-9]+", "-", name.lower()).strip("-")
    p = f"{PROJECT}/04_outputs/runs/{datetime.date.today().isoformat()}_{slug}"
    os.makedirs(p, exist_ok=True)
    return p
RUN = run_dir(NOTEBOOK_NAME)
print("Drive layout ready under", PROJECT)
print("This run saves to:", RUN)

In [ ]:
# Install this repo's package (private repo: needs GH_TOKEN in Colab Secrets, read access to damagesar-bench)
from google.colab import userdata
GH = userdata.get("GH_TOKEN")
!git clone -q https://{GH}@github.com/ThinukaLehan/damagesar-bench.git /content/damagesar-bench 2>/dev/null || (cd /content/damagesar-bench && git pull -q)
!pip install -q -e /content/damagesar-bench
COMMIT = !git -C /content/damagesar-bench rev-parse --short HEAD
COMMIT = COMMIT[0]
# A running Colab kernel does not see an editable install until a restart: put src/ on the path.
import sys, importlib
if "/content/damagesar-bench/src" not in sys.path:
    sys.path.insert(0, "/content/damagesar-bench/src")
importlib.invalidate_caches()
import damagesar_bench
print("damagesar-bench at commit", COMMIT, "->", damagesar_bench.__file__)

In [ ]:
# GPU and the p03 run
import glob, os, torch
assert torch.cuda.is_available(), "No GPU: Runtime -> Change runtime type -> T4 GPU"
P03 = sorted(glob.glob(f"{PROJECT}/04_outputs/runs/*_p03-controls-and-imagenet-v0/bench_run"))
assert P03, "missing the p03 run folder: run p03 first"
P03 = P03[-1]
print("GPU:", torch.cuda.get_device_name(0), "| p03:", P03)

In [ ]:
# BRIGHT post-event SAR only (~2 GB): labels come from the building table
import zipfile
from huggingface_hub import hf_hub_download
ROOT = "/content/bright"
post = f"{ROOT}/post-event"
if not (os.path.isdir(post) and len(os.listdir(post)) > 4000):
    p = hf_hub_download("Kullervo/BRIGHT", "post-event.zip", repo_type="dataset", local_dir="/content/zips")
    with zipfile.ZipFile(p) as z:
        z.extractall(ROOT)
print(len(os.listdir(post)), "post-event tiles")

In [ ]:
# Building table from p01: development + secondary events only (confirmatory stay locked)
import datetime, gc
import pandas as pd
from damagesar_bench.config import load_events
events = load_events("/content/damagesar-bench/configs/events.yaml")
B = pd.read_parquet(f"{PROJECT}/01_data/derived/buildings.parquet")
B["damaged"] = B["damaged"].astype("boolean")
use = B[B["role"].isin(["development", "secondary"])].copy()
del B
FEAT_DIR = f"{PROJECT}/02_models/features/{datetime.date.today().isoformat()}_finetune"
os.makedirs(FEAT_DIR, exist_ok=True)
print(use[use["excluded_reason"].isna()].groupby("event").size().to_string())

In [ ]:
# 1. Building crops (32 m, 112 px) for every kept building of the development and secondary events
from damagesar_bench.baselines.finetune import FinetuneConfig, building_crops, leave_one_event_out
from damagesar_bench.encoders.extract import read_bright_tile
CFG = FinetuneConfig()
table, crops = building_crops(use, lambda t: read_bright_tile(ROOT, t), CFG, log=print)
print(crops.shape, round(crops.nbytes / 1e9, 2), "GB")

In [ ]:
# 2. Fine-tune one model per held-out development event and score it
dev = events.keys_with_role("development")
FS = leave_one_event_out(table, crops, dev, CFG, log=print)
FS.to_parquet(f"{RUN}/fold_scores.parquet", index=False)

In [ ]:
# 3. Per-event AUROC and event-level comparisons with the frozen probes (same events, same statistics)
import json
from damagesar_bench.stats.compare import compare, event_metrics
M = event_metrics(FS, events, n_boot=2000, seed=0)
M.to_csv(f"{RUN}/event_metrics.csv", index=False)
ALL = pd.concat([M, pd.read_csv(f"{P03}/event_metrics.csv")], ignore_index=True)
FT = ("vitb16_finetuned", CFG.pooling)
rows = []
for label, b in [("fine-tuned vs DINOv3 probe", ("dinov3_vitb16@native", "outline")),
                 ("fine-tuned vs ImageNet probe", ("imagenet_vitb16@native", "outline"))]:
    c = compare(ALL, FT, b, events=dev)
    rows.append({"label": label, "estimate": c.interval.estimate, "lower": c.interval.lower,
                 "upper": c.interval.upper, "verdict": c.verdict.meaning, "sign_flip_p": c.sign_flip.p_value,
                 "gains": json.dumps({k: round(float(v), 6) for k, v in c.gains.items()})})
C = pd.DataFrame(rows); C.to_csv(f"{RUN}/comparisons.csv", index=False)
pv = ALL[ALL["pooling"].isin(["outline", CFG.pooling])].pivot_table(index="encoder", columns="event", values="auroc")
pv["mean"] = pv.mean(axis=1)
print(pv.round(3).to_string()); print(C.round(4).to_string(index=False))

In [ ]:
# File the executed notebook (with outputs) into this run's folder and 03_notebooks/pipeline/
import json
from google.colab import _message
# Ask the browser for the notebook; it can time out (busy tab, large outputs). Results above are already saved.
nb = None
for attempt in range(3):
    try:
        reply = _message.blocking_request("get_ipynb", timeout_sec=120)
        nb = (reply or {}).get("ipynb")
    except Exception as ex:
        print("attempt", attempt + 1, "failed:", ex)
    if nb:
        break
if not nb:
    print("Could not copy the executed notebook automatically (the results above are already saved).")
    print("Use File -> Save a copy in Drive, then move the copy into", RUN)
else:
    stamp = datetime.date.today().isoformat()
    for target in (f"{RUN}/{NOTEBOOK_NAME}_executed.ipynb",
                   f"{PROJECT}/03_notebooks/pipeline/{stamp}_{NOTEBOOK_NAME}.ipynb"):
        with open(target, "w") as fh:
            json.dump(nb, fh)
        print("saved", target)
    print("run folder:", sorted(os.listdir(RUN)))